Why can we dump batchnorm params after training? We'll see its because a * (Wx + y) + b => (aW)x + (ay + b)

Its a massive effort to do so I didn't bother with it

In [13]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('/Users/cono/code/learning_ml/src/ml_learning/makemore_conor/names.txt', 'r').read().splitlines()

In [3]:
len(words)

32033

In [4]:
# We had S/E special tokens, but now just "."
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [14]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [15]:
torch.manual_seed(0)


# -------------------------------------------------------
# Original network
# -------------------------------------------------------

class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(20, 64)
        self.bn1 = nn.BatchNorm1d(64)

        self.fc2 = nn.Linear(64, 64)
        self.bn2 = nn.BatchNorm1d(64)

        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.fc3(x)
        return x


model = MLP()

## Normal w/o folding

In [16]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(300):
    x = torch.randn(128, 20)
    y = torch.randint(0, 10, (128,))

    logits = model(x)
    loss = F.cross_entropy(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Training complete")

Training complete


In [17]:
def fold_bn_into_linear(linear, bn):
    """
    Returns a new Linear layer whose output is identical to

        bn(linear(x))

    during inference.
    """

    assert not bn.training

    W = linear.weight.detach().clone()
    b = linear.bias.detach().clone() if linear.bias is not None else \
        torch.zeros(W.size(0), device=W.device)

    gamma = bn.weight.detach()
    beta = bn.bias.detach()

    mean = bn.running_mean.detach()
    var = bn.running_var.detach()
    eps = bn.eps

    # scale factor
    s = gamma / torch.sqrt(var + eps)

    # scale each output row
    W_folded = W * s[:, None]

    b_folded = (b - mean) * s + beta

    new_linear = nn.Linear(
        linear.in_features,
        linear.out_features,
        bias=True,
    )

    new_linear.weight.data.copy_(W_folded)
    new_linear.bias.data.copy_(b_folded)

    return new_linear

In [18]:
model.eval()

# Fold both BatchNorms
fc1_folded = fold_bn_into_linear(model.fc1, model.bn1)
fc2_folded = fold_bn_into_linear(model.fc2, model.bn2)

In [20]:
import copy
class FoldedMLP(nn.Module):
    def __init__(self, fc1, fc2, fc3):
        super().__init__()

        self.fc1 = fc1
        self.fc2 = fc2
        self.fc3 = copy.deepcopy(fc3)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


folded_model = FoldedMLP(
    fc1_folded,
    fc2_folded,
    model.fc3,
)

folded_model.eval()

FoldedMLP(
  (fc1): Linear(in_features=20, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)

In [21]:
x = torch.randn(32, 20)

with torch.no_grad():
    y1 = model(x)
    y2 = folded_model(x)

print("Max abs difference:",
      (y1 - y2).abs().max().item())

print("Outputs identical?",
      torch.allclose(y1, y2, atol=1e-6))

Max abs difference: 3.2782554626464844e-07
Outputs identical? True
